# Silver

 notebooku və SQL bağlantısını hazırlayırıq.

In [293]:
import sys
from pathlib import Path
from sqlalchemy import text, Unicode

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src" / "db.py").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("src/db.py tapılmadı")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
from src.db import get_engine

engine = get_engine()

with engine.connect() as conn:
    db_name = conn.execute(text("SELECT DB_NAME()")).scalar()

print("Python və SQL Server hazırdır:", db_name)

Python və SQL Server hazırdır: BankAnalyticsDB


## Müştəri və Kredit məlumatı cədvəli

In [294]:
# Bronze-dakı birləşdirilmiş kredit məlumatlarını oxuyuruq
import pandas as pd
with engine.connect() as conn:
    silver_loans_df = pd.read_sql(
        text("SELECT * FROM bronze.loans_enriched"),
        conn
    )

In [295]:
silver_loans_df.head()

,customer_id,age,gender,city,branch,tenure_years,credit_score,balance,salary,products_count,...,interest_rate,status,rejection_reason,defaulted,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday
0,C00001,39,Qadın,Bakı,Yasamal,2,654,3156.63,908.98,2,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
1,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,13.88,Rədd edildi,Natamam sənədlər,NaN,2026-04-03,0.0,NaN,0.0,1.0,0.0
2,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,6.67,Rədd edildi,Yüksək borc yükü,NaN,2026-05-12,0.0,NaN,0.0,1.0,0.0
3,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,14.47,Təsdiqləndi,NaN,0.0,2026-04-06,0.0,NaN,0.0,1.0,0.0
4,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,14.33,Təsdiqləndi,NaN,0.0,2026-05-22,0.0,NaN,0.0,1.0,0.0


## Datanın təmizlənməsi

In [296]:
silver_loans_df.shape

(6891, 28)

In [297]:
silver_loans_df.duplicated().sum()

np.int64(15)

In [298]:
silver_loans_df = silver_loans_df.drop_duplicates()

In [299]:
silver_loans_df.duplicated().sum()

np.int64(0)

### Aşkarlanmış 15 dublikat sətir silindi

## Boş dəyərlərə baxaq

In [300]:
silver_loans_df.isna().sum().reset_index()

,index,0
0,customer_id,0
1,age,0
2,gender,0
3,city,0
4,branch,0
5,tenure_years,0
6,credit_score,0
7,balance,0
8,salary,0
9,products_count,0


### Bir çox sətirdə yaranmış 2076 boş dəyər loans-kredit  cədvəlinin customer-müştəri cədvəlinə birləşməsindən yaranmış boş dəyərlərdir.   

### defaulted-gecikme sütunundakı 3091 boş dəyəri araşdıraq. 2076 sı birləşmədən yaranan boş dəyərlərdir. geriye qalan 1825 boş dəyərin hamısının rədd edilmiş kreditlərə aid olmasını gözləyirik çünki rədd olunmuş kreditlərdə gecikmə olmamalıdır. 

In [301]:
silver_loans_df.groupby("status")["defaulted"].apply(lambda x: x.isna().sum())

status
Rədd edildi    1776
Təsdiqləndi       0
rədd edildi      49
təsdiqləndi       0
Name: defaulted, dtype: int64

### 1826 bos deyerin hamisi redd edilen kreditlere aiddir. bu normaldir. 

### Rejection_reason 5142 boş dəyəri araşdıraq. boş dəyərlər yalnız təsdiqdənmiş kreditlər üçün olmalıdı.

In [302]:
silver_loans_df.groupby('status')['rejection_reason'].apply(
    lambda x: x.isna().sum()
).reset_index(name='bos_deyer_sayi')

,status,bos_deyer_sayi
0,Rədd edildi,89
1,Təsdiqləndi,2914
2,rədd edildi,2
3,təsdiqləndi,61


In [303]:
silver_loans_df.loc[
    (silver_loans_df['status'].isin(['Rədd edildi', 'rədd edildi'])) &
    (silver_loans_df['rejection_reason'].isna()),
    ['loan_id', 'customer_id', 'status', 'rejection_reason']
].head(10)


,loan_id,customer_id,status,rejection_reason
63,L003335,C00043,Rədd edildi,NaN
152,L003432,C00114,Rədd edildi,NaN
161,L002875,C00119,Rədd edildi,NaN
304,L000327,C00231,Rədd edildi,NaN
330,L000588,C00248,Rədd edildi,NaN
429,L003695,C00311,Rədd edildi,NaN
447,L004125,C00320,Rədd edildi,NaN
544,L001171,C00386,Rədd edildi,NaN
590,L003744,C00423,Rədd edildi,NaN
640,L000019,C00459,Rədd edildi,NaN


### Rədd edilmiş kreditlərdə 91 boş dəyər var. bunları silməyək və "Səbəb qeyd olunmayıb" kimi qeyd edek

In [304]:
silver_loans_df.loc[
    (silver_loans_df['status'].isin(['Rədd edildi', 'rədd edildi'])) &
    (silver_loans_df['rejection_reason'].isna()),
    'rejection_reason'
] = 'Səbəb qeyd olunmayıb'

### Interest_rate de bos olan 2126 deyere baxaq

In [306]:
silver_loans_df[silver_loans_df["interest_rate"].isna()]["status"].value_counts(dropna=False)

status
NaN            2076
Təsdiqləndi      34
Rədd edildi      15
rədd edildi       1
Name: count, dtype: int64

### 2076 sətir birləşmədən yaranan boş dəyərlərdir. 34-ü təsdiqlənib — bunlarda faiz dərəcəsi yazılmalıdır. 16-sı rədd edilib — bunlarda faizin boş qalması normal ola bilər.

### kredit novlerinin medianlarini tapaq ve bu deyerlere uygun bosluqlari dolduraq

In [307]:
silver_loans_df.groupby("product_type")["interest_rate"].median()

product_type
Avtomobil krediti    11.00
Biznes krediti       12.13
İpoteka               8.09
İstehlak krediti     14.06
Name: interest_rate, dtype: float64

In [308]:
mask = silver_loans_df["status"] == "təsdiqləndi"
medians = silver_loans_df.groupby("product_type")["interest_rate"].transform("median")

silver_loans_df.loc[mask, "interest_rate"] = silver_loans_df.loc[mask, "interest_rate"].fillna(medians)

yerde qalan bos deyerlerin sayina baxaq

In [309]:
silver_loans_df["interest_rate"].isna().sum()

np.int64(2126)

In [310]:
silver_loans_df.groupby("is_holiday")["holiday_name"].apply(lambda x: x.isna().sum())

is_holiday
0.0    4763
1.0       0
Name: holiday_name, dtype: int64

bayram gunlerinde bos deyer yoxdur

In [311]:
silver_loans_df.shape


(6876, 28)

In [312]:
silver_loans_df.dtypes.reset_index()

,index,0
0,customer_id,str
1,age,int64
2,gender,str
3,city,str
4,branch,str
5,tenure_years,int64
6,credit_score,int64
7,balance,float64
8,salary,float64
9,products_count,int64


Sütun tiplərini dəyişək

In [313]:
silver_loans_df["application_date"] = pd.to_datetime(
    silver_loans_df["application_date"],
    errors="coerce"
)

silver_loans_df["merge_date"] = pd.to_datetime(
    silver_loans_df["merge_date"],
    errors="coerce"
)

In [314]:
sutunlar = [
    "age", "credit_score", "products_count",
    "has_card", "is_active", "churned"
]

silver_loans_df[sutunlar] = silver_loans_df[sutunlar].astype("Int64")

In [315]:
silver_loans_df["defaulted"] = silver_loans_df["defaulted"].astype("Int64")
silver_loans_df["tenure_years"] = silver_loans_df["tenure_years"].astype("Int64")

In [316]:
pd.set_option("display.max_rows", None)

silver_loans_df.dtypes.to_frame("Data tipi")

,Data tipi
customer_id,str
age,Int64
gender,str
city,str
branch,str
tenure_years,Int64
credit_score,Int64
balance,float64
salary,float64
products_count,Int64


### Məlumatları kiçik hərfə keçirək

In [317]:
string_cols = silver_loans_df.select_dtypes(include=["object", "string"]).columns

silver_loans_df[string_cols] = silver_loans_df[string_cols].apply(
    lambda col: col.str.lower()
)

In [318]:
silver_loans_df.loc[
    (silver_loans_df['status'] == 'rədd edildi') &
    (silver_loans_df['rejection_reason'].isna()),
    ['loan_id', 'customer_id', 'status', 'rejection_reason']
]

,loan_id,customer_id,status,rejection_reason


In [319]:
### Boşluqları silək

In [320]:
string_cols = silver_loans_df.select_dtypes(include=["object", "string"]).columns

for col in string_cols:
    silver_loans_df[col] = silver_loans_df[col].str.strip()

### Cədvəli sql silver svhemasina yazaq

In [321]:
from sqlalchemy import Unicode
silver_loans_df.to_sql(
    "loans", engine, schema="silver",
    if_exists="replace", index=False,
    dtype={c: Unicode(255) for c in silver_loans_df.select_dtypes(include=["object", "string"]).columns if c != "merge_date"}
)

68

In [322]:
silver_loans_df.to_csv(
    "../../data/bank_data/silver_loans.csv",
    index=False,
    encoding="utf-8-sig"
)

# transactions_enriched

Bronze dan transaction enriched faylini oxuyaq

In [323]:
# Bronze-dakı birləşdirilmiş kredit məlumatlarını oxuyuruq
with engine.connect() as conn:
    silver_transactions_df = pd.read_sql(
        text("SELECT * FROM bronze.transactions_enriched"),
        conn
    )
   

In [324]:
silver_transactions_df.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,...,holiday_name,is_weekend,is_workday,is_pre_holiday,rate_currency,rate_base,rate,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,T0040264,C02529,3/27/2026,19:55:59,9.91,AZN,Alış-veriş,POS,34,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,10.9,35.5,25.1
1,T0040265,C04195,3/27/2026,19:56:53,794.63,AZN,Nağdlaşdırma,ATM,45,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,10.9,35.5,25.1
2,T0040266,C03812,3/27/2026,19:57:15,120.90,AZN,Köçürmə,Mobil tətbiq,40,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,10.9,35.5,25.1
3,T0040267,C00782,3/27/2026,19:57:17,13.81,AZN,Alış-veriş,POS,48,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,10.0,14.3,13.8
4,T0040268,C02866,3/27/2026,19:57:21,36.76,AZN,Alış-veriş,POS,41,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,13.1,14.3,11.0


In [325]:
silver_transactions_df.shape

(77396, 32)

In [326]:
silver_transactions_df.duplicated().sum()

np.int64(0)

In [327]:
silver_transactions_df.isna().sum()

transaction_id             0
customer_id                0
date                       0
time                       0
amount                     0
currency                   0
category                   0
channel                    0
age                        0
gender                     0
city                       0
branch                     0
tenure_years               0
credit_score               0
balance                    0
salary                     0
products_count             0
has_card                   0
is_active                  0
churned                    0
merge_date                 0
is_holiday                 0
holiday_name           68014
is_weekend                 0
is_workday                 0
is_pre_holiday             0
rate_currency          72100
rate_base              72100
rate                   72100
temperature_2m_mean        0
precipitation_sum          0
wind_speed_10m_max         0
dtype: int64

In [328]:
silver_transactions_df.groupby("currency")["rate"].apply(
    lambda x: x.isna().sum()
)

currency
AZN    72100
EUR        0
USD        0
Name: rate, dtype: int64

Bu normaldır. Çünki valyuta məzənnəsi cədvəlimizdə USD və EUR üçün məzənnələr var, AZN üçün isə yoxdur.

AZN-in AZN-ə məzənnəsi həmişə 1-dir. Ona görə bu boşluqları 1 ilə doldura bilərik.

In [329]:
mask = silver_transactions_df["currency"] == "AZN"
silver_transactions_df.loc[mask, "rate"] = 1

In [330]:
silver_transactions_df["rate"].isna().sum()

np.int64(0)

In [331]:
silver_transactions_df.groupby("is_holiday")["holiday_name"].apply(
    lambda x: x.isna().sum())

is_holiday
0    68014
1        0
Name: holiday_name, dtype: int64

bayram gunlerinde bos deyerler yoxdur

In [332]:
silver_transactions_df.isna().sum()[lambda x: x > 0]

holiday_name     68014
rate_currency    72100
rate_base        72100
dtype: int64

In [333]:
mask = silver_transactions_df["currency"] == "AZN"

silver_transactions_df.loc[mask, "rate_currency"] = "AZN"
silver_transactions_df.loc[mask, "rate_base"] = "AZN"

silver_transactions_df.isna().sum()[lambda x: x > 0]

holiday_name    68014
dtype: int64

In [334]:
silver_transactions_df.dtypes.to_frame("Data tipi")

,Data tipi
transaction_id,str
customer_id,str
date,str
time,str
amount,float64
currency,str
category,str
channel,str
age,int64
gender,str


In [335]:
silver_transactions_df["date"] = pd.to_datetime(
    silver_transactions_df["date"],
    format="%m/%d/%Y"
)

silver_transactions_df["merge_date"] = pd.to_datetime(
    silver_transactions_df["merge_date"]
)

In [336]:
silver_transactions_df[["date", "merge_date"]].dtypes

date          datetime64[us]
merge_date     datetime64[s]
dtype: object

melumatlari kiçik hərfə çevirək

In [337]:
string_cols = silver_transactions_df.select_dtypes(
    include=["object", "string"]
).columns

silver_transactions_df[string_cols] = silver_transactions_df[string_cols].apply(
    lambda col: col.map(
        lambda x: x.lower() if isinstance(x, str) else x
    )
)

evvelde ve axirdakı boşluqları silək

In [338]:
silver_transactions_df = silver_transactions_df.apply(
    lambda col: col.map(
        lambda x: x.strip() if isinstance(x, str) else x
    )
)

In [339]:
silver_transactions_df.to_sql(
    "transactions", engine, schema="silver",
    if_exists="replace", index=False,
    dtype={c: Unicode(255) for c in silver_transactions_df.select_dtypes(include=["object", "string"]).columns if c != "merge_date"}
)

46

In [340]:
silver_transactions_df.to_csv(
    "../../data/bank_data/silver_transactions.csv",
    index=False,
    encoding="utf-8-sig"
)

In [341]:
with engine.connect() as conn:
    silver_tables = pd.read_sql(
        text("""
            SELECT TABLE_NAME
            FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = 'silver'
            ORDER BY TABLE_NAME
        """),
        conn
    )

display(silver_tables)

,TABLE_NAME
0,loans
1,transactions
